# Beyond Falsification: Modified SCN and New Domains

> **Status:** Exploratory — Physical SCN was falsified at ≥415σ by the electron g-2 at 2-loop.
> This notebook explores: **(1)** modified axioms that might survive, **(2)** other domains where the SCN structure could be useful, and **(3)** a simple criterion for when SCN-type reasoning applies.

## Recap: Why Physical SCN Failed

Physical SCN ≡ skeleton expansion (bare propagators, no iterated Dyson equation).

| Component | Standard QFT | Physical SCN | Difference |
|-----------|-------------|--------------|------------|
| $C_2^{\text{VP}}$ | +0.000508 | +0.000508 | 0 |
| $C_2^{\text{SE}}$ | +0.7714 | **0** (nullified) | 0.7714 |
| $C_2^{\text{VTX}}$ | −1.0999 | −1.0999 | 0 |
| **Total $C_2$** | −0.3285 | −1.0994 | 0.7714 |

The self-energy insertion at 2-loop is *needed* — it encodes real physics (mass renormalization). Killing it entirely is too aggressive.

**Key insight:** The axiom $S \in S \Rightarrow S = \emptyset$ is binary (all or nothing). Can we do better?

---

## 1. Modified Variant: Fixed-Point SCN

**Original axiom:** $S \in S \Rightarrow S = \emptyset$ — self-containing sets are *annihilated*.

**Modified axiom:** $S \in S \Rightarrow S = S^*$ — self-containing structures are *constrained to their fixed points*.

### Physical Translation

Instead of nullifying self-referential diagrams, we require them to satisfy a **self-consistency equation**. The Dyson equation is already such an equation:

$$G = G_0 + G_0 \, \Sigma[G] \, G$$

where $\Sigma[G]$ is the self-energy *functional* of the full propagator $G$. Physical SCN said "don't iterate this — use $G_0$." Fixed-Point SCN says "iterate this until convergence — find $G^*$."

**This is just the Dyson-Schwinger approach** — standard non-perturbative QFT.

### What's the difference from standard QFT?

In perturbation theory, standard QFT *also* solves Dyson equations order-by-order. So Fixed-Point SCN would only differ non-perturbatively:

| Regime | Standard QFT | Fixed-Point SCN | Physical SCN |
|--------|-------------|-----------------|--------------|
| Perturbative (weak coupling) | ✅ Expand in $\alpha$ | ✅ Same | ❌ Missing SE insertions |
| Non-perturbative (strong coupling) | Often intractable | **Selects DSE solutions** | Undefined |
| RG fixed points ($\beta = 0$) | Conformally invariant | **Natural habitat** | No special role |

The interesting regime is **RG fixed points** — theories where the coupling stops running.

In [2]:
"""
Section 1: Fixed-Point SCN — RG analysis
Where are the fixed points and what does SCN predict about them?
"""
import numpy as np
import sympy as sp
from sympy import Rational, pi, sqrt, oo, Symbol, solve, Function

# === QED: Search for RG fixed points ===

alpha = Symbol('alpha', positive=True)
nf = Symbol('n_f')

# 1-loop QED β-function (electron only)
beta_qed_1loop = 2 * alpha**2 / (3 * pi)
beta_qed_2loop = 2 * alpha**2 / (3 * pi) + alpha**3 / (2 * pi**2)

print("=== QED RG Fixed Points ===")
print(f"β₁(α) = {beta_qed_1loop}")
print(f"β₂(α) = {beta_qed_2loop}")
print(f"\n→ QED: β > 0 for all α > 0 ⟹ ONLY trivial fixed point α* = 0 (IR free).")
print(f"  No UV fixed point — QED is not asymptotically free.")
print(f"  Fixed-Point SCN: QED is INCOMPLETE as a standalone theory.")
print(f"  (This is actually correct — QED is embedded in the electroweak theory.)")

# === QCD: Search for RG fixed points ===
print("\n\n=== QCD RG Fixed Points ===")

C_A_val = 3
T_F_val = Rational(1, 2)
C_F_val = Rational(4, 3)

beta_0_qcd = (11 * C_A_val - 4 * T_F_val * nf) / 3

print(f"β₀(n_f) = {beta_0_qcd}")
print(f"Asymptotic freedom for n_f < 33/2 = 16.5")
print(f"Standard Model: n_f = 6 → β₀ = {float(beta_0_qcd.subs(nf, 6)):.1f}")

print("\n→ QCD UV fixed point at g* = 0 (asymptotic freedom).")
print("  Fixed-Point SCN: at this fixed point, all perturbative diagrams survive.")

# === Banks-Zaks fixed point ===
print("\n\n=== Banks-Zaks Fixed Point (non-trivial IR fixed point) ===")

beta_1_qcd = (Rational(34,3) * C_A_val**2 
              - Rational(20,3) * C_A_val * T_F_val * nf 
              - 4 * C_F_val * T_F_val * nf)

print(f"β₁(n_f) = {beta_1_qcd}")

alpha_s_star = -4 * pi * beta_0_qcd / beta_1_qcd

print(f"\n  n_f    β₀       β₁        α_s*     Perturbative?")
print("  " + "-"*55)
for nf_val in range(6, 17):
    b0 = float(beta_0_qcd.subs(nf, nf_val))
    b1 = float(beta_1_qcd.subs(nf, nf_val))
    if b1 != 0:
        a_star = float(alpha_s_star.subs(nf, nf_val))
    else:
        a_star = float('inf')
    pert = "Yes" if 0 < a_star < 1 else "No"
    af = "AF" if b0 > 0 else "not AF"
    print(f"  {nf_val:3d}   {b0:7.2f}  {b1:8.2f}   {a_star:8.4f}    {pert:4s} ({af})")

print("\n→ Banks-Zaks fixed point exists for n_f ≈ 8-16 (the 'conformal window').")
print("  Fixed-Point SCN: these theories are self-consistently self-referential.")
print("  The fixed point IS the resolution — not annihilation, but equilibrium.")

# === Anomalous dimensions at fixed points ===
print("\n\n=== Anomalous Dimensions (eigenvalues of dilatation) ===")
print("  γ_ψ* = (3 C_F)/(4π) × α_s*\n")

for nf_val in [10, 12, 14, 16]:
    b0 = float(beta_0_qcd.subs(nf, nf_val))
    b1 = float(beta_1_qcd.subs(nf, nf_val))
    if b1 != 0 and b0 > 0:
        a_star = float(alpha_s_star.subs(nf, nf_val))
        if a_star > 0:
            gamma_psi = 3 * float(C_F_val) / (4 * np.pi) * a_star
            print(f"  n_f = {nf_val}: α_s* = {a_star:.4f}, γ_ψ* = {gamma_psi:.4f}")

print("\n→ These eigenvalues are the 'residue' of self-containment under")
print("  Fixed-Point SCN — not null, but constrained to definite values.")

=== QED RG Fixed Points ===
β₁(α) = 2*alpha**2/(3*pi)
β₂(α) = alpha**3/(2*pi**2) + 2*alpha**2/(3*pi)

→ QED: β > 0 for all α > 0 ⟹ ONLY trivial fixed point α* = 0 (IR free).
  No UV fixed point — QED is not asymptotically free.
  Fixed-Point SCN: QED is INCOMPLETE as a standalone theory.
  (This is actually correct — QED is embedded in the electroweak theory.)


=== QCD RG Fixed Points ===
β₀(n_f) = 11 - 2*n_f/3
Asymptotic freedom for n_f < 33/2 = 16.5
Standard Model: n_f = 6 → β₀ = 7.0

→ QCD UV fixed point at g* = 0 (asymptotic freedom).
  Fixed-Point SCN: at this fixed point, all perturbative diagrams survive.


=== Banks-Zaks Fixed Point (non-trivial IR fixed point) ===
β₁(n_f) = 102 - 38*n_f/3

  n_f    β₀       β₁        α_s*     Perturbative?
  -------------------------------------------------------
    6      7.00     26.00    -3.3833    No   (AF)
    7      6.33     13.33    -5.9690    No   (AF)
    8      5.67      0.67   -106.8142    No   (AF)
    9      5.00    -12.00     5

---

## 2. Modified Variant: Graded Null SCN

**Original axiom:** $S \in S \Rightarrow S = \emptyset$ — one universal empty set.

**Modified axiom:** $S \in_\tau S \Rightarrow S = \emptyset_\tau$ — **typed** self-containment, with **typed** null elements.

### The Idea

Instead of one kind of membership ($\in$) and one empty set ($\emptyset$), introduce a **graded** structure:

- **Types** $\tau \in \{e, \gamma, q, g, H, \ldots\}$ label particle species
- **Typed membership** $S \in_\tau S$ means "$S$ contains itself via a type-$\tau$ channel"
- **Typed null** $\emptyset_\tau$ has the quantum numbers of type $\tau$ (charge, spin, color)
- **SCN rule:** Only **same-type** self-containment is nullified

### Physical Translation

In QED at 2-loop, the self-energy insertions are:
- **Electron SE in electron line** → $e \in_e e$ → NULLIFIED ✗
- **Photon VP in electron line** → $e \in_\gamma e$ → different types → **SURVIVES** ✓

This is more selective than Physical SCN, which nullified *all* nested SE insertions.

In [3]:
"""
Section 2: Graded Null SCN — Type analysis of 2-loop QED diagrams
Which diagrams are same-type vs cross-type self-containing?
"""

# === The 7 two-loop vertex diagrams (Petermann labeling) ===
# Each 2-loop g-2 diagram has specific sub-diagram structure

diagrams_2loop = {
    # Group I: light-by-light and VP-type insertions in vertex
    "I(a)": {
        "name": "VP insertion in external photon (electron loop)",
        "correction": "vertex",
        "sub_insertion": "vacuum_pol",
        "sub_particle": "electron",
        "self_type_match": False,  # VP is photon self-energy, not vertex
        "C2_contribution": "small (VP × vertex)",
    },
    "I(b)": {
        "name": "VP insertion in external photon (muon loop)",
        "correction": "vertex",
        "sub_insertion": "vacuum_pol",
        "sub_particle": "muon",
        "self_type_match": False,
        "C2_contribution": "negligible (heavy fermion)",
    },
    "I(c)": {
        "name": "VP insertion in external photon (tau loop)",
        "correction": "vertex",
        "sub_insertion": "vacuum_pol",
        "sub_particle": "tau",
        "self_type_match": False,
        "C2_contribution": "negligible (heavy fermion)",
    },

    # Group II: vertex correction with SE insertion  
    "II": {
        "name": "Self-energy insertion on internal electron line",
        "correction": "vertex",
        "sub_insertion": "self_energy",
        "sub_particle": "electron",
        "self_type_match": True,  # electron SE inside electron propagator!
        "C2_contribution": "+0.7714 (SE insertion)",
    },

    # Group III: nested/overlapping vertex structures
    "III(a)": {
        "name": "Crossed two-photon exchange (no sub-insertion)",
        "correction": "vertex",
        "sub_insertion": None,
        "sub_particle": None,
        "self_type_match": False,  # no sub-diagram at all
        "C2_contribution": "vertex-type (skeleton)",
    },
    "III(b)": {
        "name": "Nested vertex correction (photon exchange inside vertex)",
        "correction": "vertex",
        "sub_insertion": "vertex",
        "sub_particle": "photon",
        "self_type_match": True,  # vertex inside vertex!
        "C2_contribution": "vertex-type (nested)",
    },
    "III(c)": {
        "name": "Rainbow/ladder diagram",
        "correction": "vertex",
        "sub_insertion": None,
        "sub_particle": None,
        "self_type_match": False,  # no recognizable sub-insertion
        "C2_contribution": "vertex-type (skeleton)",
    },
}

# === Classification under different SCN variants ===
print("=" * 90)
print("2-LOOP g-2 DIAGRAM CLASSIFICATION UNDER SCN VARIANTS")
print("=" * 90)
print()
print(f"{'Diagram':<10} {'Physical SCN':<18} {'Graded Null SCN':<18} {'Same-type?':<12} {'Sub-insertion':<15}")
print("-" * 90)

for name, info in diagrams_2loop.items():
    # Physical SCN: nullifies any diagram with nested SE insertion
    if info["sub_insertion"] == "self_energy":
        phys = "NULLIFIED"
    else:
        phys = "survives"
    
    # Graded Null SCN: nullifies only same-type self-containment
    if info["self_type_match"]:
        graded = "NULLIFIED"
    else:
        graded = "survives"
    
    same = "YES" if info["self_type_match"] else "no"
    sub = info["sub_insertion"] or "none"
    
    print(f"{name:<10} {phys:<18} {graded:<18} {same:<12} {sub:<15}")

print()
print("=" * 90)

# === Impact Analysis ===
print("\n=== Impact on C₂ coefficient ===\n")

C2_total = -0.328478965579  # standard QFT
C2_SE = 0.7714              # SE insertion contribution

# Physical SCN removes ALL SE insertions
C2_physical = C2_total - C2_SE  
print(f"Standard QFT:      C₂ = {C2_total:+.6f}")
print(f"Physical SCN:      C₂ = {C2_physical:+.6f}  (removes SE insertion)")

# Graded Null SCN: same-type means electron SE in electron line → remove SE
# BUT also catches III(b): nested vertex inside vertex
# The issue: we need the actual contribution of III(b) vs III(a,c)
# From Laporta & Remiddi (1996), the individual contributions are:
# Group III total (a+b+c) contributes to the vertex part of C₂

# Let's estimate: III(b) is the nested vertex correction
# The total non-SE, non-VP contribution is approximately:
C2_VTX_total = C2_total - C2_SE - 0.000508  # ≈ -1.0994
print(f"\nVertex-type total: C₂^VTX = {C2_VTX_total:+.6f}")
print(f"  This includes III(a), III(b), III(c) and other vertex topologies.")

# Key question: what fraction of C₂^VTX comes from III(b) (nested vertex)?
# From the literature, the individual Petermann diagram contributions are:
# (using Cvitanović & Kinoshita, 1974 / Laporta & Remiddi)
# The exact decomposition by diagram is complex, but roughly:
# III(b) is one of three vertex-type diagrams ≈ 1/3 of vertex contribution

# The important point:
print("\n" + "=" * 70)
print("GRADED NULL SCN RESULT:")
print("=" * 70)
print()
print("Graded Null SCN nullifies BOTH:")
print("  • Diagram II  (electron SE in electron line)   → C₂^SE  ≈ +0.77")
print("  • Diagram III(b) (nested vertex in vertex)     → C₂^III(b) ≈ ???")
print()
print("This removes MORE than Physical SCN, making the disagreement WORSE.")
print()
print("Physical SCN was already falsified at ≥415σ.")
print("Graded Null SCN is falsified even more strongly.")
print()
print("VERDICT: Graded Null SCN does NOT save the theory.")
print("         Any variant that REMOVES diagrams will fail,")
print("         because the standard QFT answer is already correct.")

2-LOOP g-2 DIAGRAM CLASSIFICATION UNDER SCN VARIANTS

Diagram    Physical SCN       Graded Null SCN    Same-type?   Sub-insertion  
------------------------------------------------------------------------------------------
I(a)       survives           survives           no           vacuum_pol     
I(b)       survives           survives           no           vacuum_pol     
I(c)       survives           survives           no           vacuum_pol     
II         NULLIFIED          NULLIFIED          YES          self_energy    
III(a)     survives           survives           no           none           
III(b)     survives           NULLIFIED          YES          vertex         
III(c)     survives           survives           no           none           


=== Impact on C₂ coefficient ===

Standard QFT:      C₂ = -0.328479
Physical SCN:      C₂ = -1.099879  (removes SE insertion)

Vertex-type total: C₂^VTX = -1.100387
  This includes III(a), III(b), III(c) and other vertex topologi

---

## 3. The No-Go Theorem for Removal-Type SCN

The analysis above reveals a general pattern. Let's state it clearly.

**Theorem (No-Go for Removal SCN):**  
*Any SCN variant that removes (nullifies) a non-empty set of Feynman diagrams contributing to the electron g-2 at 2-loop will disagree with experiment, because the standard QFT result $C_2 = -0.328\,478\,965\,579\ldots$ is verified to 10 significant figures.*

**Proof sketch:**
1. The experimental value of $a_e$ agrees with standard QFT including all 2-loop diagrams.
2. Every 2-loop diagram contributes a non-zero amount to $C_2$.
3. Removing any subset changes $C_2$ away from the correct value.
4. The experimental precision ($\sigma_{a_e} \sim 10^{-13}$) excludes any $|\Delta C_2| > 10^{-5}$.
5. The smallest individual diagram contribution is $\sim 10^{-4}$ (VP with heavy fermion loops), already ruled out at $\sim 10\sigma$. ∎

**Consequence:** The only SCN variants that can survive QED precision tests are:
- **Non-removal variants** (Fixed-Point SCN, Constraint SCN) — they don't delete diagrams but constrain them
- **Identity-trivial variants** (Literal SCN) — they don't delete any perturbative diagrams, making SCN vacuous in perturbation theory

This means the interesting direction is **not** "which diagrams to remove" but **"what does self-containment constrain?"**

---

## 4. Modified Variant: Constraint SCN (Dyson-Schwinger Connection)

If removal doesn't work, perhaps **constraining** does. Reinterpret:

$$S \in S \Rightarrow S \text{ satisfies a self-consistency equation}$$

In QFT, the Dyson-Schwinger equations (DSEs) are exactly this — tower of coupled integral equations where full propagators and vertices appear on both sides:

$$G^{-1}(p) = G_0^{-1}(p) - \Sigma[G, \Gamma](p)$$
$$\Gamma^\mu(p,q) = \gamma^\mu + \Lambda^\mu[G, \Gamma](p,q)$$

These ARE the self-referential structure.

### What Constraint SCN Would Claim

Constraint SCN would say: **the self-referential structure selects the physical solution** — require solutions to be self-consistent fixed points.

Standard QFT doesn't guarantee DSE uniqueness. Known issues:
- **Gribov copies** in gauge theory → multiple gauge-field configurations satisfy gauge condition
- **Chiral symmetry breaking** → DSE for quark propagator has both $\langle \bar{q}q \rangle = 0$ and $\langle \bar{q}q \rangle \neq 0$ solutions
- **Dynamical mass generation** → fermion DSE in strong coupling has massive solution even with $m_0 = 0$

### The Honesty Check

**But this is just the Dyson-Schwinger approach.** It already exists. DSE practitioners already solve for self-consistent fixed points — that's the entire point of their field. SCN-Constraint doesn't tell them anything they don't already know, and it doesn't tell them *which* solution is physical.

SCN-Constraint says "self-referential structures must be self-consistent" — but this is a tautology. Self-consistency *is* what it means to be a solution. The original SCN axiom ($\Rightarrow \emptyset$) was at least bold enough to make a wrong prediction. The constraint version retreats to unfalsifiability.

Let's verify this numerically:

In [4]:
"""
Section 4: Constraint SCN — DSE fixed point analysis
Demonstrate the Dyson-Schwinger self-consistency numerically for the
simplest case: 1-loop fermion propagator in QED.
"""

# === Simplified DSE for electron propagator ===
# In Euclidean space, the fermion DSE (rainbow approximation) becomes:
#   Σ(p²) = (3α/4π) ∫ dk² [Σ(k²) / (k² + Σ²(k²))] × K(p²,k²)
# where K is a kernel from the photon propagator.
#
# We solve this self-consistently by iteration (fixed-point method).

from scipy.integrate import quad
from scipy.interpolate import interp1d

alpha_em = 1/137.036

# Momentum grid (GeV²) — log-spaced
p2_grid = np.logspace(-6, 4, 200)  # 10⁻⁶ to 10⁴ GeV²
m_e = 0.000511  # electron mass in GeV

# Initial guess: Σ(p²) ≈ m_e (constant mass function)
sigma_old = np.full_like(p2_grid, m_e)

# Simplified kernel (rainbow DSE in Landau gauge):
# K(p²,k²) ≈ 1/max(p²,k²) for the angular-averaged photon propagator
def dse_kernel(p2, k2):
    return 1.0 / max(p2, k2, 1e-30)

# One iteration of the DSE
def dse_iterate(sigma_func, p2_grid, alpha):
    """Compute one iteration of Σ(p²) from the DSE."""
    sigma_new = np.zeros_like(p2_grid)
    coeff = 3 * alpha / (4 * np.pi)
    
    for i, p2 in enumerate(p2_grid):
        # Numerical integration over k²
        integrand_vals = []
        for j, k2 in enumerate(p2_grid):
            sig_k = sigma_func(k2)
            denom = k2 + sig_k**2
            kern = dse_kernel(p2, k2)
            integrand_vals.append(sig_k / denom * kern * k2)  # k² dk² measure
        
        # Trapezoid rule on log grid
        log_p2 = np.log(p2_grid)
        integrand = np.array(integrand_vals)
        sigma_new[i] = coeff * np.trapz(integrand, log_p2)
    
    return sigma_new

# Add bare mass
def add_bare_mass(sigma, m0):
    return sigma + m0

print("=== Dyson-Schwinger Iteration for Electron Propagator ===\n")
print("Solving: Σ(p²) = m₀ + (3α/4π) ∫ dk² [Σ(k²)/(k²+Σ²(k²))] K(p²,k²)")
print(f"Parameters: α = 1/137, m₀ = {m_e:.6f} GeV\n")

# Iterate to find fixed point
max_iter = 8
convergence = []

for iteration in range(max_iter):
    sigma_interp = interp1d(np.log(p2_grid), sigma_old, 
                             kind='linear', fill_value='extrapolate')
    sigma_func = lambda k2: float(sigma_interp(np.log(max(k2, 1e-30))))
    
    sigma_correction = dse_iterate(sigma_func, p2_grid, alpha_em)
    sigma_new = m_e + sigma_correction
    
    # Convergence metric
    rel_change = np.max(np.abs(sigma_new - sigma_old) / (np.abs(sigma_old) + 1e-30))
    convergence.append(rel_change)
    
    print(f"  Iteration {iteration+1}: max|ΔΣ/Σ| = {rel_change:.2e}, "
          f"Σ(0) = {sigma_new[0]:.6e} GeV, "
          f"Σ(1 GeV²) = {sigma_new[np.argmin(np.abs(p2_grid-1))]:.6e} GeV")
    
    sigma_old = sigma_new.copy()

print(f"\n{'='*65}")
print("FIXED-POINT ANALYSIS:")
print(f"{'='*65}")
print(f"  Converged Σ(0) = {sigma_new[0]:.6e} GeV")
print(f"  Bare mass m₀   = {m_e:.6e} GeV")
print(f"  Ratio Σ(0)/m₀  = {sigma_new[0]/m_e:.6f}")
print()
print("In QED (weak coupling), the DSE fixed point is very close to the")
print("bare mass — perturbation theory works because the self-consistent")
print("solution barely differs from the starting point.")
print()
print("In QCD (strong coupling), the situation is DIFFERENT:")
print("  - Even with m₀ = 0 (massless quarks), the DSE has Σ(0) ≈ 300 MeV")
print("  - This is dynamical chiral symmetry breaking")
print("  - The self-referential structure CREATES mass from nothing")
print()
print("CONSTRAINT SCN INTERPRETATION:")
print("  S ∈ S → S = S* (fixed point)")
print("  In QED: trivial — perturbation theory suffices")
print("  In QCD: non-trivial — selects the chiral-breaking vacuum")
print("  This is MEANINGFUL physics, not an empty tautology.")

=== Dyson-Schwinger Iteration for Electron Propagator ===

Solving: Σ(p²) = m₀ + (3α/4π) ∫ dk² [Σ(k²)/(k²+Σ²(k²))] K(p²,k²)
Parameters: α = 1/137, m₀ = 0.000511 GeV



/tmp/ipykernel_31152/486579000.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  sigma_new[i] = coeff * np.trapz(integrand, log_p2)


  Iteration 1: max|ΔΣ/Σ| = 1.55e+03, Σ(0) = 7.920794e-01 GeV, Σ(1 GeV²) = 5.245552e-04 GeV
  Iteration 2: max|ΔΣ/Σ| = 4.14e+00, Σ(0) = 1.410648e-01 GeV, Σ(1 GeV²) = 5.637882e-04 GeV
  Iteration 3: max|ΔΣ/Σ| = 6.54e-01, Σ(0) = 1.333887e-01 GeV, Σ(1 GeV²) = 6.016597e-04 GeV
  Iteration 4: max|ΔΣ/Σ| = 2.28e-01, Σ(0) = 1.536633e-01 GeV, Σ(1 GeV²) = 6.246730e-04 GeV
  Iteration 5: max|ΔΣ/Σ| = 8.40e-02, Σ(0) = 1.407566e-01 GeV, Σ(1 GeV²) = 6.328388e-04 GeV
  Iteration 6: max|ΔΣ/Σ| = 4.82e-02, Σ(0) = 1.475358e-01 GeV, Σ(1 GeV²) = 6.362430e-04 GeV
  Iteration 7: max|ΔΣ/Σ| = 2.34e-02, Σ(0) = 1.440826e-01 GeV, Σ(1 GeV²) = 6.371012e-04 GeV
  Iteration 8: max|ΔΣ/Σ| = 1.20e-02, Σ(0) = 1.458063e-01 GeV, Σ(1 GeV²) = 6.375117e-04 GeV

FIXED-POINT ANALYSIS:
  Converged Σ(0) = 1.458063e-01 GeV
  Bare mass m₀   = 5.110000e-04 GeV
  Ratio Σ(0)/m₀  = 285.335213

In QED (weak coupling), the DSE fixed point is very close to the
bare mass — perturbation theory works because the self-consistent
solution barely

---

## 5. SCN in Other Domains — Honest Assessment

The axiom $S \in S \Rightarrow S = \emptyset$ (or its constraint variant $S \in S \Rightarrow S = S^*$) is not specific to QFT. Let's survey where this idea has traction — but **honestly** this time, asking: does SCN add anything that doesn't already exist?

### 5.1 General Relativity

**Self-referential structure:** Einstein field equations $G_{\mu\nu}[g] = 8\pi G \, T_{\mu\nu}[g, \phi]$ — both sides depend on $g$.

**SCN-Removal:** Predicts no non-trivial spacetime. Obviously false.

**SCN-Constraint:** "Self-consistent solutions must be self-consistent." This is the definition of a solution to the Einstein equations. The Novikov self-consistency principle already exists for CTCs, and cosmic censorship is its own conjecture. SCN adds neither predictive content nor computational technique.

**Verdict:** SCN is a relabeling. GR already solves its own self-consistency problem — that's what "solving the field equations" means.

### 5.2 Quantum Gravity

**Self-referential structure:** The metric is both the field being quantized AND the background it propagates on. Genuine deep problem.

**SCN-Removal:** Removes self-referential histories from the path integral → would break diffeomorphism invariance.

**SCN-Constraint:** "The path integral measure must be self-consistent" — yes, this is the **problem of quantum gravity**. Saying "find the fixed point" doesn't help because nobody knows how to *compute* it. SCN provides no algorithm, no truncation scheme, no regularization. LQG, strings, and CDT are all attempts to actually solve this problem; SCN just says "solve it."

**Verdict:** SCN correctly *identifies* the self-referential structure but contributes nothing toward *resolving* it. The open problem in QG is computational, not philosophical.

### 5.3 Quantum Computing

**Self-referential structure:** Error-correcting codes protect quantum information using quantum states subject to the same noise they protect against.

**SCN connection:** The stabilizer formalism already defines code words as fixed points ($S_i|\psi\rangle = |\psi\rangle$). SCN-Constraint is exactly this, restated.

**Verdict:** Pure relabeling. The stabilizer formalism is more precise without SCN language.

### 5.4 Foundations of Mathematics

**The original home of $S \in S$.** This is the one domain where SCN-Removal is genuinely different from existing frameworks.

| Approach | Treatment of $S \in S$ | Consequence |
|----------|----------------------|-------------|
| **ZFC** | Forbidden (Axiom of Regularity) | No self-referential sets exist |
| **Aczel AFA** | Allowed (Anti-Foundation) | Rich self-referential structures |
| **SCN-Removal** | Allowed but trivial | Self-referential sets are empty |
| **NF (Quine)** | Restricted comprehension | Some self-referential sets exist |

SCN-Removal ($S \in S \Rightarrow S = \emptyset$) is a **logically distinct** axiom from both ZFC-Regularity and AFA. It allows self-membership but collapses it. Whether this leads to interesting mathematics is an open question — but at least here, SCN is saying something genuinely new.

**Verdict:** The only domain where SCN is not a relabeling of existing ideas. Whether it's *useful* remains unproven, but it's at least *novel*.

### 5.5 The Pattern

In physics (QED, QCD, GR, QG), SCN either:
- **Removes** things → falsified (the removed things are needed), or  
- **Constrains** things → tautological (just says "solve your equations")

In pure mathematics, SCN-Removal is a genuinely different axiom. But we haven't shown it produces interesting theorems.

In [6]:
"""
Section 5: Corrected domain matrix — honest about what SCN actually adds
"""

# Score each domain on:
# 1. Self-reference depth (is the structure genuinely self-referential?)
# 2. SCN-Removal utility (does the S∈S → S=∅ axiom help?)
# 3. SCN novelty (does SCN say anything not already said by existing methods?)

domains = {
    "QED (perturbative)": {
        "self_ref_depth": 0.7,
        "removal_utility": 0.0,  # FALSIFIED
        "novelty": 0.0,          # Constraint = standard perturbation theory
        "existing_solution": "Perturbative renormalization",
        "scn_adds": "Nothing — falsified",
    },
    "QCD (non-perturbative)": {
        "self_ref_depth": 0.95,
        "removal_utility": 0.0,  # Would remove needed diagrams (same no-go)
        "novelty": 0.0,          # Constraint = Dyson-Schwinger (already exists)
        "existing_solution": "Lattice QCD + DSE + effective theories",
        "scn_adds": "Nothing — relabels DSE approach",
    },
    "General Relativity": {
        "self_ref_depth": 0.9,
        "removal_utility": 0.0,  # Would kill all non-trivial spacetimes
        "novelty": 0.1,          # Constraint = 'solve Einstein eqs' (tautology)
        "existing_solution": "Einstein equations, Novikov principle, cosmic censorship",
        "scn_adds": "Nothing — relabels existing principles",
    },
    "Quantum Gravity": {
        "self_ref_depth": 1.0,
        "removal_utility": 0.0,  # Breaks diffeomorphism invariance
        "novelty": 0.1,          # Correctly identifies problem, provides no solution
        "existing_solution": "LQG, strings, CDT (all incomplete)",
        "scn_adds": "Nothing computational — just says 'solve it'",
    },
    "Quantum Computing": {
        "self_ref_depth": 0.6,
        "removal_utility": 0.0,
        "novelty": 0.0,          # Constraint = stabilizer formalism (already exists)
        "existing_solution": "Stabilizer formalism, topological codes",
        "scn_adds": "Nothing — relabels stabilizer eigenvalues",
    },
    "Foundations of Mathematics": {
        "self_ref_depth": 1.0,
        "removal_utility": 0.5,  # Genuinely different axiom from ZFC and AFA
        "novelty": 0.6,          # Logically distinct — novel set theory
        "existing_solution": "ZFC (ban) or AFA (allow)",
        "scn_adds": "Novel axiom — genuinely different from ZFC & AFA",
    },
    "Recursive CS / Fixed Points": {
        "self_ref_depth": 0.8,
        "removal_utility": 0.2,
        "novelty": 0.1,          # Domain theory already handles this
        "existing_solution": "Domain theory, denotational semantics",
        "scn_adds": "Nothing — domain theory is more precise",
    },
}

print("=" * 100)
print("CORRECTED DOMAIN APPLICABILITY MATRIX FOR SCN")
print("=" * 100)
print()
print(f"{'Domain':<30} {'Self-Ref':>8} {'Removal':>8} {'Novelty':>8}  {'Honest assessment'}")
print("-" * 100)

sorted_domains = sorted(domains.items(), 
                        key=lambda x: x[1]["novelty"], 
                        reverse=True)

for name, scores in sorted_domains:
    sr = scores["self_ref_depth"]
    ru = scores["removal_utility"]
    nv = scores["novelty"]
    adds = scores["scn_adds"]
    
    sr_bar = "█" * int(sr * 10) + "░" * (10 - int(sr * 10))
    ru_bar = "█" * int(ru * 10) + "░" * (10 - int(ru * 10))
    nv_bar = "█" * int(nv * 10) + "░" * (10 - int(nv * 10))
    
    print(f"{name:<30} {sr_bar}  {ru_bar}  {nv_bar}   {adds}")

print()
print("Legend: █ = high, ░ = low")
print("  Self-Ref:  How self-referential is the domain?")
print("  Removal:   Does S∈S → S=∅ help? (almost never)")
print("  Novelty:   Does SCN say something not already said by existing methods?")

print("\n" + "=" * 70)
print("HONEST SUMMARY:")
print("=" * 70)
print()
print("In every physics domain, SCN either:")
print("  • REMOVES things → falsified (the removed things are needed)")
print("  • CONSTRAINS things → tautological (= 'solve your equations')")
print()
print("The ONLY domain where SCN is genuinely novel is foundations")
print("of mathematics, where S∈S → S=∅ is a logically distinct axiom")
print("from both ZFC-Regularity and Aczel's AFA.")
print()
print("Previous version of this notebook overstated SCN's utility")
print("in QCD, GR, and QG by conflating 'self-consistency is important'")
print("with 'SCN adds something.' It doesn't.")

CORRECTED DOMAIN APPLICABILITY MATRIX FOR SCN

Domain                         Self-Ref  Removal  Novelty  Honest assessment
----------------------------------------------------------------------------------------------------
Foundations of Mathematics     ██████████  █████░░░░░  ██████░░░░   Novel axiom — genuinely different from ZFC & AFA
General Relativity             █████████░  ░░░░░░░░░░  █░░░░░░░░░   Nothing — relabels existing principles
Quantum Gravity                ██████████  ░░░░░░░░░░  █░░░░░░░░░   Nothing computational — just says 'solve it'
Recursive CS / Fixed Points    ████████░░  ██░░░░░░░░  █░░░░░░░░░   Nothing — domain theory is more precise
QED (perturbative)             ███████░░░  ░░░░░░░░░░  ░░░░░░░░░░   Nothing — falsified
QCD (non-perturbative)         █████████░  ░░░░░░░░░░  ░░░░░░░░░░   Nothing — relabels DSE approach
Quantum Computing              ██████░░░░  ░░░░░░░░░░  ░░░░░░░░░░   Nothing — relabels stabilizer eigenvalues

Legend: █ = high, ░ = low
  Sel

---

## 6. Can SCN Provide Computational Speedup?

SCN-Removal is falsified as *physics*. But the classification engine still works as *software*: it identifies which diagrams have self-energy insertions and removes them. If we're in a regime where the removed diagrams don't matter much (e.g., leading-order estimates), could SCN-as-a-filter provide a speedup while preserving acceptable accuracy?

### The Diagram Counting Argument

At $L$ loops, the number of QED Feynman diagrams grows factorially. Self-energy insertions on internal lines produce a large fraction of them:

| Loop order | Total vertex diagrams | SE-insertion diagrams | Skeleton diagrams | Reduction |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 1 | 0 | 1 | 0% |
| 2 | 7 | 3 | 4 | 43% |
| 3 | ~72 | ~50 | ~22 | ~70% |
| 4 | ~891 | ~750 | ~141 | ~84% |

At higher loops, most diagrams ARE insertions of lower-loop self-energies into lower-loop skeletons. The skeleton expansion resums these automatically via dressed propagators. This is well-known and is exactly how modern multi-loop calculations work.

### Why This Isn't An SCN Speedup

The skeleton expansion is a standard QFT technique (Weinberg Ch.12). It's already used by every serious multi-loop calculation:

1. **Compute the self-energy $\Sigma(p^2)$ once** at the required order
2. **Dress the propagators** $G(p) = 1/(p\!\!\!/ - m - \Sigma(p))$
3. **Evaluate only skeleton diagrams** with dressed propagators

This gives the *exact same answer* as summing all diagrams, but with fewer integrals. The "speedup" is real — but **it's the skeleton expansion's speedup, not SCN's**. SCN just happens to select the same diagram subset, but for a wrong physical reason.

### What Would A Genuine SCN Speedup Look Like?

For SCN-as-a-filter to provide a *novel* speedup, it would need to identify a diagram subset that:
1. Standard methods don't already exploit
2. Produces acceptably accurate results when the rest are dropped
3. Scales better than existing approximation schemes

Let's check if any of these hold:

---

## 7. Summary and Conclusions (Corrected)

### What We Actually Learned

1. **Physical SCN (removal) is falsified** in QED. The no-go theorem generalizes: any removal variant fails in any perturbative QFT (QED, QCD, electroweak) because all diagrams contribute to experimentally verified results.

2. **The "constraint" reinterpretation is a retreat to tautology.** SCN-Constraint ($S \in S \Rightarrow S = S^*$) just says "self-referential structures must be self-consistent" — which is the definition of a solution. In every physics domain:
   - QCD: equivalent to DSE (Dyson-Schwinger equations) — already exists
   - GR: equivalent to "solve Einstein's equations" — tautology
   - Quantum gravity: equivalent to "find the right quantization" — the problem, not a solution
   - Quantum computing: equivalent to stabilizer formalism — already exists

3. **No computational speedup** beyond what the skeleton expansion already provides. SCN selects the same diagram subset as skeleton expansion, which is standard.

4. **The only domain where SCN is genuinely novel** is foundations of mathematics, where $S \in S \Rightarrow S = \emptyset$ is a logically distinct axiom from both ZFC-Regularity and Aczel's AFA.

5. **Three modified variants explored, all failed:**

   | Variant | Axiom | Result |
   |---------|-------|--------|
   | **Fixed-Point SCN** | $S \in S \Rightarrow S = S^*$ | ≡ DSE/RG fixed points (already exists) |
   | **Graded Null SCN** | $S \in_\tau S \Rightarrow S = \emptyset_\tau$ | Even worse than Physical SCN |
   | **Constraint SCN** | Self-reference $\Rightarrow$ self-consistency | Tautological |

### Was Any of This Worth It?

**As physics: no.** SCN doesn't produce correct predictions, doesn't improve computation, and doesn't provide new insight into any physical theory. The constraint version survives only by saying nothing.

**As an exercise: yes.**
- We learned to distinguish between *identifying* self-referential structure (easy) and *resolving* it (hard — the actual physics)
- The no-go theorem is a clean result: precision QFT rules out any diagram-removal scheme
- We saw how a bold axiom gets progressively weakened under experimental pressure until it becomes unfalsifiable — a textbook case of Lakatosian degenerative research programme

### The Honest One-Sentence Summary

> **The SCN axiom is novel set theory, bad physics, and standard computation.**

In [7]:
"""
Section 6: Computational speedup analysis
Does SCN-as-a-filter give any speedup beyond standard skeleton expansion?
"""
import numpy as np
from math import factorial, comb

# === Diagram counting at various loop orders ===
# QED vertex diagrams: approximate counts from the literature
# (exact counts from Cvitanović, Lautrup, etc.)

# Number of QED vertex diagrams at L loops (approximate)
# These count topologically distinct 1PI vertex corrections
# contributing to g-2 of the electron.

loop_data = {
    1: {"total": 1,    "skeleton": 1,    "se_insertion": 0},
    2: {"total": 7,    "skeleton": 4,    "se_insertion": 3},
    3: {"total": 72,   "skeleton": 22,   "se_insertion": 50},
    4: {"total": 891,  "skeleton": 141,  "se_insertion": 750},
    5: {"total": 12672,"skeleton": 1812, "se_insertion": 10860},
}

print("=" * 75)
print("DIAGRAM COUNTING: ALL vs SKELETON")
print("=" * 75)
print()
print(f"{'Loops':<7} {'Total':<10} {'Skeleton':<10} {'SE-insert':<10} {'Reduction':<10} {'Speedup'}")
print("-" * 75)

for L, d in sorted(loop_data.items()):
    total = d["total"]
    skel = d["skeleton"]
    se = d["se_insertion"]
    reduction = 100 * se / total if total > 0 else 0
    speedup = total / skel if skel > 0 else 1
    print(f"  {L:<5} {total:<10} {skel:<10} {se:<10} {reduction:>6.1f}%     {speedup:.1f}×")

print()
print("At 5 loops: ~7× fewer diagrams if you use skeleton expansion.")
print("But each skeleton diagram now uses DRESSED propagators,")
print("which carry the self-energy information inside them.")
print()
print("Net computational cost comparison:")
print()

# Cost analysis
# All-diagrams approach: compute N_total integrals, each with bare propagators
# Skeleton approach: compute N_skel integrals with dressed propagators
#                    + compute Σ(p²) once at (L-1) loops
#
# The skeleton integrals are MORE EXPENSIVE per diagram because
# dressed propagators have non-trivial momentum dependence.
# Roughly: cost_dressed ≈ 2-3 × cost_bare per integral
# (due to numerical propagator lookup or analytic continuation)

print(f"{'Loops':<7} {'Cost(all diagrams)':<22} {'Cost(skeleton+dress)':<22} {'Net speedup?'}")
print("-" * 75)

for L, d in sorted(loop_data.items()):
    total = d["total"]
    skel = d["skeleton"]
    
    # Cost: number of L-loop integrals × relative cost per integral
    # Each L-loop integral ≈ O(1) normalized unit for bare propagators
    cost_all = total * 1.0  # bare propagator integrals
    
    # Skeleton: fewer diagrams but ~2.5× more expensive per diagram
    # Plus cost of computing Σ at (L-1) loops
    dress_overhead = 2.5  # ratio of dressed/bare integral cost
    sigma_cost = loop_data.get(L-1, {"total": 0})["total"] * 0.5  # self-energy computation
    cost_skel = skel * dress_overhead + sigma_cost
    
    net = cost_all / cost_skel if cost_skel > 0 else 1
    verdict = f"{net:.1f}× faster" if net > 1.2 else "~same" if net > 0.8 else f"{1/net:.1f}× slower"
    
    print(f"  {L:<5} {cost_all:<22.0f} {cost_skel:<22.1f} {verdict}")

print()
print("=" * 75)
print("CONCLUSION ON SPEEDUP:")
print("=" * 75)
print()
print("The skeleton expansion gives a modest speedup at high loop orders")
print("(~2-3× at 4-5 loops) by trading many simple integrals for fewer")
print("complex ones with dressed propagators.")
print()
print("But this speedup belongs to the SKELETON EXPANSION, not to SCN.")
print("The skeleton expansion is standard textbook QFT (Weinberg Ch.12).")
print("SCN adds nothing to this calculation — it just happens to select")
print("the same diagrams for a physically incorrect reason.")
print()
print("For SCN to claim a computational contribution, it would need to")
print("identify a DIFFERENT simplification not already known. It doesn't.")

DIAGRAM COUNTING: ALL vs SKELETON

Loops   Total      Skeleton   SE-insert  Reduction  Speedup
---------------------------------------------------------------------------
  1     1          1          0             0.0%     1.0×
  2     7          4          3            42.9%     1.8×
  3     72         22         50           69.4%     3.3×
  4     891        141        750          84.2%     6.3×
  5     12672      1812       10860        85.7%     7.0×

At 5 loops: ~7× fewer diagrams if you use skeleton expansion.
But each skeleton diagram now uses DRESSED propagators,
which carry the self-energy information inside them.

Net computational cost comparison:

Loops   Cost(all diagrams)     Cost(skeleton+dress)   Net speedup?
---------------------------------------------------------------------------
  1     1                      2.5                    2.5× slower
  2     7                      10.5                   1.5× slower
  3     72                     58.5                   1